In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import muon as mu

In [ ]:
perturb_cite_rna = ad.read_h5ad("data/perturb_citeseq/perturb_cite_seq_rna_processed.h5ad")
perturb_cite_protein = ad.read_h5ad("data/perturb_citeseq/perturb_cite_seq_protein_processed.h5ad")
perturb_cite_rna

In [ ]:
perturb_cite_rna.obs

In [ ]:
import os, json, re, warnings

def parse_sgRNA_to_gene(sgrna_series):
    """
    Parse sgRNA column: 'IRF4_3' -> 'IRF4', 'SDCBP_3' -> 'SDCBP'.
    Strip trailing _N guide number. Mark non-targeting and nan as 'control'.
    """
    def _parse(val):
        if pd.isna(val) or str(val).lower() == 'nan':
            return 'control'
        s = str(val).strip()
        # Non-targeting guides
        if any(kw in s.upper() for kw in ['NON-GENE', 'NO_SITE', 'NON_TARGET', 'SCRAMBLE']):
            return 'control'
        # Strip trailing _N (guide number): e.g. IRF4_3 -> IRF4, A2M_1 -> A2M
        # But be careful with gene names that contain underscores (rare but possible)
        match = re.match(r'^(.+)_(\d+)$', s)
        if match:
            return match.group(1)
        return s
    
    return sgrna_series.apply(_parse)

In [ ]:
perturb_cite_rna

In [ ]:
TRAIN_RNA_PATH = "data/pbmc_citeseq/rna_final_processed.h5ad"
TRAIN_PROT_PATH = "data/pbmc_citeseq/cite_seq_processed.h5ad"
PERTURB_RNA_PATH = "data/perturb_citeseq/perturb_cite_seq_rna_processed.h5ad"
PERTURB_PROT_PATH = "data/perturb_citeseq/perturb_cite_seq_protein_processed.h5ad"

train_rna = sc.read_h5ad(TRAIN_RNA_PATH)
train_prot = sc.read_h5ad(TRAIN_PROT_PATH)
train_gene_names = train_rna.var_names.tolist()
train_prot_names = train_prot.var_names.tolist()

print("Loading perturbation data...")
perturb_rna = sc.read_h5ad(PERTURB_RNA_PATH)
perturb_prot = sc.read_h5ad(PERTURB_PROT_PATH)

# common_obs = sorted(set(perturb_rna.obs_names) & set(perturb_prot.obs_names))
# perturb_rna = perturb_rna[common_obs].copy()
# perturb_prot = perturb_prot[common_obs].copy()
# print(f"  Aligned cells: {len(common_obs)}")

perturb_rna.obs['target_gene'] = parse_sgRNA_to_gene(perturb_rna.obs['sgRNA'])

moi_col = perturb_rna.obs['MOI']
is_control = perturb_rna.obs['target_gene'] == 'control'
# keep_mask = (moi_col == '1') | is_control
keep_mask = is_control
perturb_rna = perturb_rna[keep_mask].copy()
perturb_prot = perturb_prot[keep_mask].copy()
print(f"  After MOI==1 filter (+ controls): {perturb_rna.n_obs} cells")

target_genes = perturb_rna.obs['target_gene']
control_mask = target_genes == 'control'

pert_counts = target_genes[~control_mask].value_counts()
print(f"  Control cells: {control_mask.sum()}")
print(f"  Unique targeted genes: {len(pert_counts)}")


In [ ]:
perturb_rna

In [ ]:
perturb_rna.obs["target_gene"].value_counts()

In [ ]:
print(f"  Unique targeted genes: {len(pert_counts)}")

In [ ]:
perturb_cite_rna_tmp.obs[perturb_cite_rna_tmp.obs["MOI"] == "1"]

In [ ]:
perturb_cite_rna.obs["sgRNA"].value_counts()

In [ ]:
perturb_cite_protein

In [ ]:
pbmc_cite_rna = ad.read_h5ad("data/pbmc_citeseq/rna_final_processed.h5ad")
pbmc_cite_protein = ad.read_h5ad("data/pbmc_citeseq/cite_seq_processed.h5ad")
pbmc_cite_rna

In [ ]:
list(pbmc_cite_rna.var_names)

In [ ]:
len(set(pbmc_cite_rna.var_names) & set(perturb_cite_rna.var_names))

In [ ]:
set(pbmc_cite_protein.var_names) & set(perturb_cite_protein.var_names)

In [ ]:
list(pbmc_cite_protein.var_names)

In [ ]:
set(perturb_cite_protein.var_names) in set(pbmc_cite_protein.var_names)

In [ ]:
perturb_cite_protein.var_names[4]

In [ ]:
for prot in list(perturb_cite_protein.var_names):
    if prot not in list(pbmc_cite_protein.var_names):
        print(prot)

In [ ]:
list(pbmc_cite_protein.var_names)